In [27]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0):
    n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION -- each step's buffer is a 4-tone composite
#    (ratio-weighted durations). All 4 tones shift together as the chirp
#    offset sweeps from CHIRP_OFFSET_START_HZ to CHIRP_OFFSET_STOP_HZ.
#
#    ARCHITECTURE NOTE: this is deliberately back to the composite-per-step
#    + mode="periodic" design (same as your originally-proven script), NOT
#    the shared-library + active-retrigger design explored afterward.
#    Measured on your hardware: each tProc-triggered switch costs ~20ns
#    regardless of buffer size. With small shared-library buffers and many
#    repeats, total switching overhead across the sweep (585k switches x
#    20ns =~ 11.7ms) exceeded the entire 6ms sweep duration -- that's a
#    hard architectural limit, not a tuning problem. mode="periodic" avoids
#    it entirely because the DAC repeats the buffer via its own internal
#    address wrap, with NO tProc trigger (and therefore no ~20ns tax)
#    between repeats. The cost is back to composite-per-step memory
#    scaling (NUM_STEPS x composite size), which is why NUM_STEPS is
#    capped by ENV_MAXLEN via CYCLE_S below.
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ = 350e6
CHIRP_OFFSET_STOP_HZ = 0.0
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ

NUM_STEPS = 45
AMPLITUDE = 0.5

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6 + 76.25e6, -147.82e6 + 76.25e6])
TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])

BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S = 6e-3
STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
CYCLE_S = max_feasible_total_s / (NUM_STEPS + 1)

tone_fractions = TONE_RATIOS / TONE_RATIOS.sum()
tone_durations_s = tone_fractions * CYCLE_S
print(f"Composite buffer cycle: {CYCLE_S*1e9:.2f} ns, held for "
      f"{STEP_HOLD_US:.2f} us per chirp step ({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
for i, (bt, dur, r) in enumerate(zip(BASE_TONES_HZ, tone_durations_s, TONE_RATIOS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> requested {dur*1e9:.2f} ns")

def build_multitone_buffer(chirp_offset_hz, phase0):
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur in zip(BASE_TONES_HZ, tone_durations_s):
        f = chirp_offset_hz + base_tone
        y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, [len(p) for p in i_pieces]

chirp_offsets_hz = np.linspace(CHIRP_OFFSET_START_HZ, CHIRP_OFFSET_STOP_HZ, NUM_STEPS)
maxv = soccfg.get_maxv(GEN_CH)

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase)
    idata_list.append(idata)
    qdata_list.append(qdata)

samples_per_step = len(idata_list[0])
total_samples = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0
assert min(piece_lens) >= 3 * samps_per_clk, (
    f"Smallest tone slice is only {min(piece_lens)} samples "
    f"({min(piece_lens)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3. "
    f"Reduce NUM_STEPS, or make TONE_RATIOS less extreme."
)

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER -- same ratio-weighted 4-tone composite, at the FINAL chirp
#     frequency, phase-continuous with the last sweep step. Reuses the same
#     CYCLE_S/tone_durations_s as every sweep step -- this is exactly why
#     CYCLE_S divides by (NUM_STEPS + 1) above: one extra reserved slot.
# -----------------------------------------------------------------------------
trap_idata, trap_qdata, phase, trap_piece_lens = build_multitone_buffer(CHIRP_OFFSET_STOP_HZ, phase)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or CYCLE_S."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM: sweep as before, then ONE final pulse on the concatenated trap
#    buffer with mode="periodic" -- hardware loops it forever on its own.
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])
        self.pulse(ch=res_ch, t='auto')
        self.sync_all(step_cycles)

    def update(self):
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

    def make_program(self):
        """Standard RAveragerProgram sweep (expts x reps), with a single
        trapping pulse appended after the loop -- no loop construct needed
        for trapping, since mode="periodic" makes the hardware repeat it
        on its own once triggered."""
        p = self
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, self.cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, self.cfg['reps'] - 1)
        p.label("LOOP_J")
        p.body()
        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, self.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')
        p.update()
        p.loopnz(0, rii, "LOOP_I")

        # --- trapping: one pulse, periodic buffer, then end(). The DAC keeps
        # cycling the 4 concatenated tones forever regardless of what the
        # tProc does after this -- including after it hits end() and halts.
        res_ch = self.cfg["res_ch"]
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=self.cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,'''}}}}}}}}}]
    '''
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
prog.run(soc)
print(f"Running on hardware -- {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
Composite buffer cycle: 137.68 ns, held for 133.33 us per chirp step (6.000 ms total sweep)
  tone 0 (-76.2 MHz offset): ratio 0.337 -> requested 46.40 ns
  tone 1 (-0.0 MHz offset): ratio 0.167 -> requested 22.99 ns
  tone 2 (+46.7 MHz offset): ratio 0.288 -> requested 39.65 ns
  tone 3 (+71.6 MHz offset): ratio 0.208 -> requested 28.64 ns
Per-step composite buffer length: 1392 samples (tone slices: [464, 240, 400, 288], smallest = 15.0 fabric cycles)
Total envelope samples (sweep): 62640 / 65536 available
Trap buffer: 1392 samples (tone slices: [464, 240, 400, 288])
Total envelope samples (sweep + trap): 64032 / 65536 available
Running on hardware -- 45 sweep steps x 133.33 us = 6.000 ms sweep, then trapping (4 tones, periodic, indefinitely until soc.reset_gens()).


In [26]:
soc.reset_gens()
